# 08 · Diagnostics: the project's one-off checks

Each script here was written to answer one specific question along the way: "do the scenes look right?",
"is the model using a shortcut?", "has it memorized its training signs?". They aren't part of the main
pipeline, but they're how the project found its problems. This notebook runs them from one place.

| section | script | the question it answers | runs |
|---|---|---|---|
| 1 | `show_data.py` | Do the stored scenes look right? | always |
| 2 | `show_train_scenes.py` | What does training actually see, augmentation included? | always |
| 3 | `make_demo_figure.py` | What do right and wrong predictions look like? | always |
| 4 | `shortcut_check.py` | Does the model fire on things that aren't signs? | always |
| 5 | `probe_memorization.py` | Has it memorized training scenes, or training signs? | `RUN_FOR_REAL` |
| 6 | `mask_check_real.py` | On real windows, can it guess the class with the sign blacked out? | `RUN_FOR_REAL` |
| 7 | `p6_join.py` | Are v3a's real "false alarms" actually unlabeled signs? | `RUN_FOR_REAL` |

"Always" scripts write only into `outputs/notebooks/figures/`, so running them can't overwrite anything the
project relies on. The `RUN_FOR_REAL` ones write recorded results (under `outputs/eval/` or `results/`), so
they're off until you turn them on. Their stored results are shown either way.

**Needs:** the stored datasets and a trained run. **Writes:** `outputs/notebooks/figures/`, plus recorded
results if you turn on `RUN_FOR_REAL`.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_decoy"  # the trained run that sections 3-5 check
TRAIN_CONFIG = "configs/peak_decoy.yaml"  # the training recipe section 2 draws scenes from (must use fresh scenes)
RUN_FOR_REAL = False  # True: also run sections 5-7, which write recorded results (about 1-2 min in all)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Find the run, and set up a folder for figures plus a helper that shows a PNG file in the notebook.
from IPython.display import Image, display

require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")
RUN_DIR = find_run(RUN_NAME)
CKPT = (RUN_DIR / "best.pt").relative_to(ROOT)  # relative, as the scripts record it in their JSON
FIG = (NB_OUT / "figures").relative_to(ROOT)  # relative, so the printed commands stay short


def show(png, width=900):
    """Display a PNG (path relative to the repo root) at the given width in pixels."""
    display(Image(filename=str(ROOT / png), width=width))


print("checkpoint:", CKPT)
print("figures go to:", FIG)

## 1. The stored scenes up close (`show_data.py`)

Four pictures of a stored split. **anatomy** takes one scene apart: the image, its box, the heatmap target
the model is trained to output, and the sign's pixels. **samples** is 24 random scenes. **classes** is one
sign per class, at the size the model sees. **difficulty** sorts signs by brightness and by size, so you can
see how dark and how small they get. The script refuses `--split test`, so test scenes are never looked at.

In [ ]:
# Draw the four data figures for val (a few seconds).
run_script("show_data", "--config", "configs/baseline.yaml", "--split", "val", "--out", FIG, "--seed", 0)

In [ ]:
# Show the anatomy and samples figures (classes and difficulty are in the same folder).
show(f"{FIG}/data_anatomy_val.png")
show(f"{FIG}/data_samples_val.png")

## 2. What training sees (`show_train_scenes.py`)

From v1.5b on, training scenes are made fresh every epoch, with the signs randomly rotated, warped, recolored
and partly covered. This draws the first 24 training scenes of epoch 1 exactly as training builds them, plus
one sign drawn 12 times through the augmentation. If a warp looked implausible, or the box stopped hugging the
sign, it would show up here.

In [ ]:
# Draw epoch 1's first 24 training scenes and one sign's 12 augmented versions (a few seconds).
import yaml

run_script("show_train_scenes", "--config", TRAIN_CONFIG, "--epoch", 1, "--out", FIG)
train_run = yaml.safe_load((ROOT / TRAIN_CONFIG).read_text())["train"]["run_name"]  # names the output files
show(f"{FIG}/train_scenes_{train_run}.png")
show(f"{FIG}/sign_variants_{train_run}.png")

## 3. Right and wrong, side by side (`make_demo_figure.py`)

Scores all of val, then draws 8 random correct scenes and 4 random wrong ones. It's the figure in the
project README, made here for whichever run you pick. The `--out` is given on purpose: the script's default
would overwrite the README's `docs/demo_val.png`.

In [ ]:
# Draw 8 right + 4 wrong val predictions for RUN_NAME (~10 s).
run_script("make_demo_figure", "--checkpoint", CKPT, "--n-right", 8, "--n-wrong", 4, "--seed", 0,
           "--out", f"{FIG}/demo_val.png")
show(f"{FIG}/demo_val.png", width=700)

## 4. Does it fire on non-signs? (`shortcut_check.py`)

The test that started v3a. It builds 1,000 new backgrounds and renders each in several versions that differ
only in what's pasted on: nothing, a real sign, or a non-sign rectangle pasted exactly the way signs are:

* `bg_patch`: a crop of another photo, the same idea as v3a.1's decoys,
* `shuffled_sign`: a sign's pixels scrambled, so the same colors with no shape,
* `flat`: a solid patch in a sign's average color,
* `other_sign`: a second real sign, which *should* fire (a positive control).

"Fires" here means the heatmap peak is above **0.5**, a fixed cutoff, not the run's τ from notebook 06.
A model that fires on `bg_patch` has learned "pasted rectangle = sign".

In [ ]:
# Run the shortcut check on RUN_NAME with 1,000 backgrounds (~20 s); it prints a table and draws a panel.
run_script("shortcut_check", "--checkpoint", CKPT, "--n", 1000, "--seed", 12345,
           "--panel-out", f"{FIG}/shortcut_check.png")
show(f"{FIG}/shortcut_check.png")

**Reading the panel:** each row is one background. Columns come in pairs, the scene and then its heatmap in
red: *sign + decoy*, *decoy only*, *nothing pasted*. Green box = real sign, red box = decoy. A good model glows
inside the green box only.

**Reading the table:** `peak>0.5` is the share of scenes where the model fires. For `peak_decoy` the decoy rows
fire on 1.5–2.6% of scenes, close to the 1.0% for empty scenes. Try `RUN_NAME = "peak_fresh_aug"` (v1.5b, which
never saw a scene without a sign): it fires on 12% of `bg_patch`, 15% of `shuffled_sign` and 27% of `flat`
decoys. That's the shortcut v3a and v3a.1 were built to remove.

## 5. Memorization (`probe_memorization.py`)

Two ways a model can look good without having learned what we want:

* **scene memorization**: it remembers the exact training images. That can't happen with fresh scenes
  (v1.5b on), since no training scene is ever repeated.
* **sign memorization**: it knows the particular physical signs it trained on, not signs in general.
  GTSRB has several photos of each physical sign, and val uses different physical signs from training.

The probe scores 5,000 *new* scenes built from **training** signs, and compares with val (new signs). The
difference is the **sign gap**. Classes with few training signs (≤ 12) are reported separately, because
that's where memorizing is easiest.

In [ ]:
# Stored probe results for every run that has one: joint accuracy on new scenes with training signs vs val.
EVAL = ROOT / "outputs/eval"
print(f"{'run':34} {'train signs':>11} {'val':>7} {'sign gap':>9}  {'weak classes: train signs':>25} {'val':>6}")
for path in sorted(EVAL.glob("*/best_probe.json")):
    p = read_json(path)
    print(f"{p['run_id']:34} {p['fresh_train_signs']['joint_acc']:11.4f} {p['val']['joint_acc']:7.4f} "
          f"{p['sign_gap_joint']:9.4f}  {p['weak_class_acc']['fresh']:25.4f} {p['weak_class_acc']['val']:6.4f}")

For v1.5b (`peak_fresh_aug`) the sign gap is about 1 point overall, and about 2.5 points on the weak classes:
some sign memorization, concentrated where there are few signs to learn from.

In [ ]:
# Probe RUN_NAME (writes outputs/eval/<run>/best_probe.json; ~1 min). Only runs when RUN_FOR_REAL = True.
run_script("probe_memorization", "--checkpoint", CKPT, "--n", 5000, "--seed", 987654, enabled=RUN_FOR_REAL)

## 6. Masking the sign on real windows (`mask_check_real.py`)

Blacks out the sign's box in every real_dev window and scores the model again. If class accuracy stayed well
above the **majority-class rate** (the score from always guessing the most common class) with the sign gone,
something else in the window would be giving the answer away: the pole, the street, the photo it came from.

It takes no options and always checks the v1.5b seed 0 checkpoint (`peak_fresh_aug`), whatever `RUN_NAME` is.

In [ ]:
# Mask the signs and re-score (writes outputs/eval/mask_check/). Only runs when RUN_FOR_REAL = True.
run_script("mask_check_real", enabled=RUN_FOR_REAL)

## 7. Are real "false alarms" really signs? (`p6_join.py`)

GTSDB labels only its 43 classes, so a real no-sign window can still hold a sign of another kind (a crosswalk
sign, a no-stopping sign) or a sign too small to be labeled. If the model says "sign" there, it counts as a
false alarm, but it isn't really wrong.

Prediction P6 of v3a said that at least 10 of the 20 highest-presence real no-sign windows would be such
unlabeled signs. To keep the check honest, the 40 windows (those 20, plus 20 random ones as a control) were
labeled **from a sheet with no scores on it**, and the labels were committed *before* `p6_join.py` joined them
with the scores. The script checks that the labels file is committed and unchanged, which proves that order.

Labels: **A** = a sign outside the 43 classes, **B** = an unlabeled sign of one of the 43, **C** = sign-like
but not a sign (a light, a logo), **D** = nothing sign-like.

In [ ]:
# The stored P6 result: label counts among the 20 top windows and the 20 controls.
p6 = read_json(ROOT / "results/v3a/p6_result.json")
for group in ("top", "control"):
    g = p6[group]
    print(f"{group:8} {g['counts']}   signs (A+B): {g['sign']} of {g['n']}")
print("P6 holds (at least 10 of the top 20 are signs):", p6["p6_holds"])
show("results/v3a/p6_sheet.png", width=800)

P6 failed: 8 of the top 20 were unlabeled signs, not 10. The other 12 split evenly between sign-like things
(C) and nothing sign-like (D), so most of v3a's real false alarms are genuine mistakes. That still beats the
control group, though: 1 sign in 20 random windows vs 8 in the top 20. The model's most confident real false
alarms are unlabeled signs far more often than chance.

In [ ]:
# Re-join the committed labels with the key (rewrites results/v3a/p6_result.json). Only when RUN_FOR_REAL = True.
P6_KEY = (EVAL / find_run("peak_empty").name / "p6_key.json").relative_to(ROOT)  # the sheet was made from v3a seed 0
run_script("p6_join", "--labels", "results/v3a/p6_labels.json", "--key", P6_KEY, enabled=RUN_FOR_REAL)

## Where the other scripts live

| script | notebook |
|---|---|
| `make_dataset.py` | 01 |
| `train.py` | 03 |
| `evaluate.py` | 04 |
| `make_real_dataset.py`, `check_gtsdb_overlap.py`, `report_real.py` | 05 |
| `make_empty_dataset.py`, `evaluate_presence.py`, `seam_report.py`, `compare_presence.py` | 06 |
| `infer.py` | 07 |

`tb_demo.py` writes a fake TensorBoard run to check the dashboard works. It dates from before `train.py`
existed and isn't needed any more.

That's the whole project. Notebook 00 lists what's on disk; `docs/` holds each version's spec and results.